# Used Car Price Prediction
Reproducible modeling notebook for the supervised regression assignment.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
df = pd.read_excel("../data/Used_Car_Price_Regression_Dataset.xlsx", sheet_name="Car_Data")
X = df.drop(columns=["Car_ID", "Price_USD"])
y = df["Price_USD"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(df.shape)
print(df.isnull().sum().sum())


## Preprocessing
Categorical variables are one-hot encoded. Numerical variables are standardized for Linear Regression and passed through unchanged for Random Forest.

In [ ]:
categorical_features = ["Brand_Tier", "City_Market", "Fuel_Type", "Transmission", "Accident_History"]
numerical_features = ["Car_Age_Years", "Mileage_KM", "Engine_Size_CC", "Horsepower_HP", "Previous_Owners", "Service_History_Score"]
scaled = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features), ("num", StandardScaler(), numerical_features)])
tree = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features), ("num", "passthrough", numerical_features)])


## Linear Regression Baseline

In [ ]:
linear_pipeline = Pipeline([("preprocessor", scaled), ("model", LinearRegression())])
linear_pipeline.fit(X_train, y_train)
linear_pred = linear_pipeline.predict(X_test)
print("R2:", r2_score(y_test, linear_pred))
print("MAE:", mean_absolute_error(y_test, linear_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, linear_pred)))


## Baseline Random Forest and 5-Fold Cross-Validation

In [ ]:
rf_pipeline = Pipeline([("preprocessor", tree), ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
print("Test R2:", r2_score(y_test, rf_pred))
print("Test MAE:", mean_absolute_error(y_test, rf_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
cv = cross_validate(rf_pipeline, X_train, y_train, cv=5, scoring={"r2":"r2", "mae":"neg_mean_absolute_error", "rmse":"neg_root_mean_squared_error"}, n_jobs=-1)
print("Mean CV R2:", cv["test_r2"].mean())


## Hyperparameter Tuning

In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": [0.7, 1.0]
}
grid_search = GridSearchCV(rf_pipeline, param_grid, cv=5, scoring="r2", n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(grid_search.best_score_)


## Final Tuned Model Evaluation

In [ ]:
tuned_rf = grid_search.best_estimator_
tuned_pred = tuned_rf.predict(X_test)
print("R2:", r2_score(y_test, tuned_pred))
print("MAE:", mean_absolute_error(y_test, tuned_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, tuned_pred)))


### Final reported holdout results
Tuned Random Forest: R² = 0.9647, MAE = $1,883.02, RMSE = $2,440.04.